In [12]:
import torch 
from torch.utils.data import Dataset 
from torch.utils.data import DataLoader
from torch import nn
from torch.nn import functional as F
import pandas as pd 
import torch 
from sklearn.model_selection import train_test_split

from torch import optim
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
from src.tokenizer import Tokenizer

In [14]:
# Read the French translation file
with open('../fra.txt', 'r', encoding='utf-8') as file:
    lines = file.readlines()

# Parse the translation pairs
translation_pairs = []
for line in lines:
    line = line.strip()
    if line:
        # Split by tab (common format for translation files)
        parts = line.split('\t')
        if len(parts) >= 2:
            english = parts[0].strip()
            french = parts[1].strip()
            translation_pairs.append((english, french))

print(f"Loaded {len(translation_pairs)} translation pairs")
print("Sample translations:")
for i in range(min(5, len(translation_pairs))):
    print(f"EN: {translation_pairs[i][0]}")
    print(f"FR: {translation_pairs[i][1]}")
    print("-" * 40)

Loaded 167130 translation pairs
Sample translations:
EN: Go.
FR: Va !
----------------------------------------
EN: Hi.
FR: Salut !
----------------------------------------
EN: Run!
FR: Cours !
----------------------------------------
EN: Run!
FR: Courez !
----------------------------------------
EN: Who?
FR: Qui ?
----------------------------------------


In [ ]:
english_tokenizer = Tokenizer(lang='en')
french_tokenizer = Tokenizer(lang='fr')

for english_sentence, french_sentence in translation_pairs:
    english_tokenizer.add_words_from_sentence(english_sentence)
    french_tokenizer.add_words_from_sentence(french_sentence)

english_word_to_id = english_tokenizer.word_to_id
english_id_to_word = english_tokenizer.id_to_word
english_word_count = len(english_word_to_id)

french_word_to_id = french_tokenizer.word_to_id
french_id_to_word = french_tokenizer.id_to_word
french_word_count = len(french_word_to_id)

print(f"English vocabulary size: {english_word_count}")
print(f"French vocabulary size: {french_word_count}")

# Display some sample mappings
print("\nSample English mappings:")
for i, (word, id) in enumerate(english_word_to_id.items()):
    if i >= 5:
        break
    print(f"'{word}': {id}")

print("\nSample French mappings:")
for i, (word, id) in enumerate(french_word_to_id.items()):
    if i >= 5:
        break
    print(f"'{word}': {id}")

TypeError: Tokenizer.__init__() missing 1 required positional argument: 'lang'

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

english_sequences = []
french_sequences = []
english_pad_id = 0

for english_sentence, french_sentence in translation_pairs:
    # Convert English sentence to tensor
    encoded_english_sentence = english_tokenizer.encode_sentence(english_sentence)
    english_sequences.append(encoded_english_sentence)
    
    # Convert French sentence to tensor, adding <sos> and <eos> tokens
    encoded_french_sentence = french_tokenizer.encode_sentence(french_sentence, add_sos_eos=True)
    french_sequences.append(encoded_french_sentence)

# Pad sequences using PyTorch's pad_sequence utility
english_tensor = pad_sequence(english_sequences, batch_first=True, padding_value=0)
french_tensor = pad_sequence(french_sequences, batch_first=True, padding_value=-0)

print(f"English tensor shape: {english_tensor.shape}")
print(f"French tensor shape: {french_tensor.shape}")

# Print a couple of samples
print("\nSample 1:")
print(f"Original English: {translation_pairs[0][0]}")
print(f"Original French: {translation_pairs[0][1]}")
print(f"English tensor: {english_tensor[0][:10]}...")  # Show first 10 values
print(f"French tensor: {french_tensor[0][:10]}...")

print("\nSample 2:")
print(f"Original English: {translation_pairs[1][0]}")
print(f"Original French: {translation_pairs[1][1]}")
print(f"English tensor: {english_tensor[1][:10]}...")
print(f"French tensor: {french_tensor[1][:10]}...")

English tensor shape: torch.Size([167130, 47])
French tensor shape: torch.Size([167130, 56])

Sample 1:
Original English: Go.
Original French: Va !
English tensor: tensor([3, 0, 0, 0, 0, 0, 0, 0, 0, 0])...
French tensor: tensor([1, 3, 4, 2, 0, 0, 0, 0, 0, 0])...

Sample 2:
Original English: Hi.
Original French: Salut !
English tensor: tensor([4, 0, 0, 0, 0, 0, 0, 0, 0, 0])...
French tensor: tensor([1, 5, 4, 2, 0, 0, 0, 0, 0, 0])...


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Create lists to store sequences as tensors
english_sequences = []
french_sequences = []

english_pad_id = french_pad_id = 0


for english_sentence, french_sentence in translation_pairs:
    # Convert English sentence to tensor
    en_words = [word.lower() for word in english_sentence.split() if word.strip()]
    en_ids = [english_word_to_id.get(word, english_pad_id) for word in en_words]
    english_sequences.append(torch.tensor(en_ids, dtype=torch.long))
    
    # Convert French sentence to tensor
    fr_words = [word.lower() for word in french_sentence.split() if word.strip()]
    fr_ids = [french_word_to_id['<sos>']] + [french_word_to_id.get(word, french_pad_id) for word in fr_words] + [french_word_to_id['<eos>']]
    french_sequences.append(torch.tensor(fr_ids, dtype=torch.long))

# Pad sequences using PyTorch's pad_sequence utility
english_tensor = pad_sequence(english_sequences, batch_first=True, padding_value=0)
french_tensor = pad_sequence(french_sequences, batch_first=True, padding_value=-0)

print(f"English tensor shape: {english_tensor.shape}")
print(f"French tensor shape: {french_tensor.shape}")

# Print a couple of samples
print("\nSample 1:")
print(f"Original English: {translation_pairs[0][0]}")
print(f"Original French: {translation_pairs[0][1]}")
print(f"English tensor: {english_tensor[0][:10]}...")  # Show first 10 values
print(f"French tensor: {french_tensor[0][:10]}...")

print("\nSample 2:")
print(f"Original English: {translation_pairs[1][0]}")
print(f"Original French: {translation_pairs[1][1]}")
print(f"English tensor: {english_tensor[1][:10]}...")
print(f"French tensor: {french_tensor[1][:10]}...")


English tensor shape: torch.Size([167130, 47])
French tensor shape: torch.Size([167130, 56])

Sample 1:
Original English: Go.
Original French: Va !
English tensor: tensor([3, 0, 0, 0, 0, 0, 0, 0, 0, 0])...
French tensor: tensor([1, 3, 4, 2, 0, 0, 0, 0, 0, 0])...

Sample 2:
Original English: Hi.
Original French: Salut !
English tensor: tensor([4, 0, 0, 0, 0, 0, 0, 0, 0, 0])...
French tensor: tensor([1, 5, 4, 2, 0, 0, 0, 0, 0, 0])...


In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split

# Define a custom Dataset
class TranslationDataset(Dataset):
    def __init__(self, english_tensor, french_tensor):
        self.english_tensor = english_tensor
        self.french_tensor = french_tensor

    def __len__(self):
        return self.english_tensor.size(0)

    def __getitem__(self, idx):
        return {
            "english": self.english_tensor[idx],
            "french": self.french_tensor[idx],
        }

# Create an instance of the dataset
dataset = TranslationDataset(english_tensor, french_tensor)

# Define split sizes
total_size = len(dataset)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size

# Split the dataset
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")
print(f"Test set size: {len(test_dataset)}")

# Create DataLoaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Example of fetching one batch from the train_loader
print("\nSample batch from train_loader:")
sample_batch = next(iter(train_loader))
print(f"English batch shape: {sample_batch['english'].shape}")
print(f"French batch shape: {sample_batch['french'].shape}")

# Print the content of the sample batch
print("\nSample batch content:")
print(sample_batch)

Training set size: 133704
Validation set size: 16713
Test set size: 16713

Sample batch from train_loader:
English batch shape: torch.Size([64, 47])
French batch shape: torch.Size([64, 56])

Sample batch content:
{'english': tensor([[ 205,  681, 2583,  ...,    0,    0,    0],
        [ 793, 2335,  337,  ...,    0,    0,    0],
        [  78,   29,    0,  ...,    0,    0,    0],
        ...,
        [ 121,  544,   76,  ...,    0,    0,    0],
        [ 339,  319,  285,  ...,    0,    0,    0],
        [  16, 3105, 1426,  ...,    0,    0,    0]]), 'french': tensor([[   1, 1692, 1266,  ...,    0,    0,    0],
        [   1,   59, 9005,  ...,    0,    0,    0],
        [   1,  176,  177,  ...,    0,    0,    0],
        ...,
        [   1,   74,  178,  ...,    0,    0,    0],
        [   1, 3152, 1908,  ...,    0,    0,    0],
        [   1,   26, 1691,  ...,    0,    0,    0]])}


In [ ]:
from src.model import Transformer as Transformer2
model = Transformer2(english_word_count, french_word_count)


In [ ]:
def do_evaluation(loader, 
                  model, 
                  device,
                  criterion):
    model.eval()  # Set model to evaluation mode

    running_loss = 0.0
    total_correct = 0
    total_samples = 0
    with torch.no_grad():
        for i, batch in enumerate(loader):
            english, french = batch['english'], batch['french']
            english = english.to(device)
            french = french.to(device)

            decoder_input = french[:, :-1]
            decoder_input
            labels = french[:, 1:]
            # y = y.to(device).to(torch.float32).unsqueeze(1)

            # 1. Forward pass
            logits = model(english, decoder_input)

            logits = logits.reshape(-1, french_word_count)
            labels = labels.reshape(-1)

            # 2. Calculate Loss
            loss = criterion(logits, labels)
            running_loss += loss.item()

            # 3. Calculate Accuracy
            # Get the predicted class (index with the highest logit)
            preds = torch.argmax(logits, dim=1)

            # Count correct predictions in this batch
            total_correct += (preds == labels).sum().item()

            # Count total samples in this batch
            total_samples += labels.size(0)

    # Calculate average loss and accuracy for the entire dataset
    avg_loss = running_loss / len(loader)
    avg_accuracy = total_correct / total_samples

    return avg_loss, avg_accuracy

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

### Create 
NUM_EPOCHS = 10 
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.1,
    patience=3,
)

print("device", device)

for epoch in range(NUM_EPOCHS):
    running_loss = 0

    
    for idx, batch in enumerate(train_loader):
        english, french = batch['english'], batch['french']

        english = english.to(device)
        french = french.to(device)


        decoder_input = french[:, :-1]
        decoder_input.to(device)

        optimizer.zero_grad()
        logits = model(english, decoder_input)
        labels = french[:, 1:]

        logits = logits.reshape(-1, french_word_count)
        labels = labels.reshape(-1)

        loss = criterion(logits, labels)
        if idx % 1000 == 0:
            print(loss)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()


    epoch_loss = running_loss / len(train_loader)
    scheduler.step(epoch_loss)
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], Loss: {epoch_loss:.4f}")


train_loss = epoch_loss
test_loss, test_accuracy = do_evaluation(test_loader, model, device, criterion)
print(f"  Train Loss: {train_loss:.4f}")
print(f"  Test. Loss:  {test_loss:.4f}")
print(f"  Test Accuracy: {test_accuracy:.4f}")

device cuda
tensor(10.8377, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(2.9652, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(2.3832, device='cuda:0', grad_fn=<NllLossBackward0>)
Epoch [1/10], Loss: 3.3049
tensor(1.9835, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(2.1138, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(1.8042, device='cuda:0', grad_fn=<NllLossBackward0>)
Epoch [2/10], Loss: 1.8953
tensor(1.2456, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(1.6412, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(1.7590, device='cuda:0', grad_fn=<NllLossBackward0>)
Epoch [3/10], Loss: 1.4066
tensor(0.9575, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.9854, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(1.1321, device='cuda:0', grad_fn=<NllLossBackward0>)
Epoch [4/10], Loss: 1.1332
tensor(0.8974, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.7495, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.9268, device='cuda:0', grad_fn

In [ ]:

def encode_sentence(sentence, word_map):
    output = []
    pad_id = 0
    for word in sentence.split():
        id = word_map.get(word.lower(), pad_id)
        output.append(id)
    return torch.tensor(output)




model.eval()

sentence = "I am from New York."
encoded_sentence = encode_sentence(sentence, english_word_to_id)
# (1, Seq_Len)
encoded_sentence = encoded_sentence.to(device).unsqueeze(0) 

# Start with just <sos>
decoder_sentence = encode_sentence('<sos>', french_word_to_id)
# (1, 1)
decoder_sentence = decoder_sentence.to(device).unsqueeze(0) 

output_tokens = []

print(f"Input: {sentence}")

for _ in range(20):
    # 1. Run the model
    # logits shape: (Batch, Current_Seq_Len, Vocab_Size)
    logits = model(encoded_sentence, decoder_sentence)
    
    # 2. Focus ONLY on the last step
    # We slice [:, -1, :] to get the logits for the last token only.
    last_token_logits = logits[:, -1, :] # Shape: (Batch, Vocab_Size)
    
    # 3. Get the prediction
    probs = F.softmax(last_token_logits, dim=-1)
    next_word_id = torch.argmax(probs, dim=-1) # Shape: (Batch,) i.e., [word_id]
    
    next_word_val = next_word_id.item()
    word_string = french_id_to_word.get(next_word_val, "???")
    
    print(f"Step {_}: Predicted ID {next_word_val} ({word_string})")
    output_tokens.append(word_string)

    
    # 4. Stop if we hit <eos>
    if word_string == '<eos>':
        print("End of sentence reached.")
        break

    # 5. CRITICAL FIX: Append the new word to the history!
    # We take the current decoder_sentence and add the new ID to the end.
    # next_word_id is (1), we need to make it (1, 1) to concatenate
    next_token_tensor = next_word_id.unsqueeze(1) 
    
    decoder_sentence = torch.cat([decoder_sentence, next_token_tensor], dim=1)


print(' '.join(output_tokens))

Input: I am from New York.
Step 0: Predicted ID 26 (je)
Step 1: Predicted ID 130 (viens)
Step 2: Predicted ID 84 (de)
Step 3: Predicted ID 14877 (new)
Step 4: Predicted ID 14878 (york.)
Step 5: Predicted ID 2 (<eos>)
End of sentence reached.
je viens de new york. <eos>
